In [ ]:
# =============================================================================
# Predicting WTI Crude Oil Price Changes: A Multi-Factor Econometric Analysis
# =============================================================================
# MFIN8852.01 - Financial Econometrics
#
# Hypothesis:
#   Macroeconomic and financial market variables — including equity returns,
#   implied volatility, energy commodities, the US dollar, interest rates,
#   inventory levels, geopolitical risk, and industrial activity — contain
#   statistically significant predictive information for monthly (and weekly)
#   percentage changes in WTI crude oil prices.
#
# We employ OLS multiple regression as a baseline, then test for time-series
# pathologies (autocorrelation, ARCH effects) in the residuals.  Where
# conditional heteroskedasticity is present we re-estimate with an
# ARX-GARCH(1,1) framework so that coefficient inference is conducted under
# correct standard errors.  Both monthly and weekly frequencies are examined
# to assess robustness and determine which granularity yields better
# predictive power.
#
# Data window: December 1990 – December 2025 (normalized across all series).
# =============================================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import het_arch, het_breuschpagan, acorr_ljungbox
from statsmodels.stats.outliers_influence import variance_inflation_factor
from arch import arch_model

pd.set_option("display.max_columns", 25)
pd.set_option("display.width", 120)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["figure.figsize"] = (12, 5)

## 1. Data Loading

### 1a. Monthly Data
Load the pre-computed monthly percentage-change series from the "Combined Monthly" sheet. All series have been date-aligned to month-end observations from December 1990 through December 2025.

In [ ]:
# ── Monthly data from "Combined Monthly" sheet ──────────────────────────────
col_map = {
    0: "date",
    7: "wti_chg_pct",
    14: "vix_chg_pct",
    21: "sp500_chg_pct",
    31: "ng_chg_pct",
    39: "heatoil_chg_pct",
    47: "gld_chg_pct",
    57: "dxy_chg_pct",
    60: "tcmsy10_rate",
    64: "spr_inv_chg",
    67: "refin_util_rate",
    71: "crude_stk_excl_spr",
    75: "gpr_idx_chg",
    77: "gpr_threat_idx_chg",
    81: "us_crude_prod",
    85: "indpro",
    89: "igrea_chg_pct",
    97: "copper",
}

raw = pd.read_excel(
    "../data/data.xlsx",
    sheet_name="Combined Monthly",
    header=None,
    skiprows=3,
    usecols=list(col_map.keys()),
)
raw.columns = [col_map[c] for c in raw.columns]
monthly = raw.set_index("date")
monthly.index = pd.to_datetime(monthly.index)
monthly = monthly.apply(pd.to_numeric, errors="coerce")
monthly = monthly.loc["1990-12-01":"2025-12-01"].copy()
print(f"Monthly dataset: {monthly.shape[0]} obs x {monthly.shape[1]} variables")
print(f"Date range: {monthly.index[0].strftime('%b %Y')} – {monthly.index[-1].strftime('%b %Y')}")
monthly.head()

### 1b. Weekly Data
The "Combined Weekly" sheet is empty, so we construct weekly series by resampling daily closing prices to Friday-end weeks and computing weekly percentage returns. Variables that are only available at monthly frequency (GPR, INDPRO, IGREA, crude production) are excluded from the weekly analysis.

In [ ]:
# ── Weekly data: resample daily closing prices to weekly returns ─────────────
daily_cols = {
    0: "date",
    1: "wti_close",
    7: "vix_close",
    14: "sp500_close",
    27: "ng_close",
    33: "heatoil_close",
    39: "gld_close",
    51: "dxy_close",
    57: "copper_close",
    65: "tcmsy10_rate",
    67: "spr_inv",
    68: "refin_util_rate",
    69: "crude_stk_excl_spr",
}

daily_raw = pd.read_excel(
    "../data/data.xlsx",
    sheet_name="Combined Daily",
    header=None,
    skiprows=2,
    usecols=list(daily_cols.keys()),
)
daily_raw.columns = [daily_cols[c] for c in daily_raw.columns]
daily_raw = daily_raw.set_index("date")
daily_raw.index = pd.to_datetime(daily_raw.index)
daily_raw = daily_raw.apply(pd.to_numeric, errors="coerce")

# Forward-fill weekly-release data (SPR, refinery, crude stocks, 10yr) before resampling
# These are released weekly/monthly but only appear on release dates in the daily sheet
for col in ["tcmsy10_rate", "spr_inv", "refin_util_rate", "crude_stk_excl_spr"]:
    daily_raw[col] = daily_raw[col].ffill()

# Resample to weekly (Friday close) using last available observation
weekly_close = daily_raw.resample("W-FRI").last()

# Compute weekly percentage returns for price series
price_cols = ["wti_close", "vix_close", "sp500_close", "ng_close",
              "heatoil_close", "gld_close", "dxy_close", "copper_close"]

weekly = pd.DataFrame(index=weekly_close.index)
for col in price_cols:
    weekly[col.replace("_close", "_chg_pct")] = weekly_close[col].pct_change()

# Percentage change in level variables
weekly["spr_inv_chg"] = weekly_close["spr_inv"].pct_change()
weekly["crude_stk_excl_spr_chg"] = weekly_close["crude_stk_excl_spr"].pct_change()
weekly["tcmsy10_rate"] = weekly_close["tcmsy10_rate"]
weekly["refin_util_rate"] = weekly_close["refin_util_rate"]

# Trim to Dec 1990 – Dec 2025 and drop rows where WTI is missing
weekly = weekly.loc["1990-12-01":"2025-12-31"]
weekly = weekly.dropna(subset=["wti_chg_pct"])

# Drop columns that are still >50% NaN (insufficient daily coverage)
na_pct = weekly.isnull().mean()
sparse_cols = na_pct[na_pct > 0.50].index.tolist()
if sparse_cols:
    print(f"Dropping columns with >50% missing: {sparse_cols}")
    weekly = weekly.drop(columns=sparse_cols)

print(f"Weekly dataset: {weekly.shape[0]} obs x {weekly.shape[1]} variables")
print(f"Date range: {weekly.index[0].strftime('%Y-%m-%d')} – {weekly.index[-1].strftime('%Y-%m-%d')}")
print(f"Missing values:\n{weekly.isnull().sum()}")
weekly.head()

---
## 2. Exploratory Data Analysis (Monthly)

### 2.1 Descriptive Statistics & Missing Data

In [ ]:
# Descriptive statistics
print("=== Monthly Descriptive Statistics ===\n")
print(monthly.describe().round(4).to_string())
print(f"\n=== Missing values per column ===\n{monthly.isnull().sum()}")
print(f"\nTotal observations: {len(monthly)}")
print(f"Complete cases (no NaN in any column): {monthly.dropna().shape[0]}")

### 2.2 WTI Returns Time Series & Distribution

In [ ]:
# ── Figure 1: WTI monthly returns time series ───────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Time series of WTI returns
axes[0].plot(monthly.index, monthly["wti_chg_pct"], linewidth=0.8, color="steelblue")
axes[0].axhline(0, color="red", linestyle="--", linewidth=0.6)
axes[0].set_title("Monthly WTI Crude Oil Price Changes (%)", weight="bold")
axes[0].set_ylabel("% Change")
axes[0].grid(True, alpha=0.3)

# Rolling mean and std (12-month window)
roll_win = 12
roll_mean = monthly["wti_chg_pct"].rolling(roll_win).mean()
roll_std = monthly["wti_chg_pct"].rolling(roll_win).std()
axes[0].plot(monthly.index, roll_mean, color="red", linewidth=1.5, label=f"{roll_win}-Mo Rolling Mean")
axes[0].plot(monthly.index, roll_std, color="green", linewidth=1.5, label=f"{roll_win}-Mo Rolling Std Dev")
axes[0].legend()

# Histogram of WTI returns
axes[1].hist(monthly["wti_chg_pct"].dropna(), bins=50, color="steelblue",
             edgecolor="black", alpha=0.7, density=True)
axes[1].set_title("Distribution of Monthly WTI Returns", weight="bold")
axes[1].set_xlabel("% Change")
axes[1].set_ylabel("Density")
axes[1].grid(True, alpha=0.3)

# Overlay normal distribution for comparison
mu, sigma = monthly["wti_chg_pct"].mean(), monthly["wti_chg_pct"].std()
x = np.linspace(mu - 4*sigma, mu + 4*sigma, 200)
axes[1].plot(x, (1/(sigma*np.sqrt(2*np.pi))) * np.exp(-0.5*((x-mu)/sigma)**2),
             color="red", linewidth=2, label="Normal fit")
axes[1].legend()

plt.tight_layout()
plt.show()

# Skewness and kurtosis
from scipy.stats import jarque_bera, skew, kurtosis
wti = monthly["wti_chg_pct"].dropna()
print(f"Skewness:  {skew(wti):.4f}")
print(f"Kurtosis:  {kurtosis(wti, fisher=False):.4f}  (Normal = 3.0)")
jb_stat, jb_p = jarque_bera(wti)
print(f"Jarque-Bera: stat={jb_stat:.2f}, p-value={jb_p:.4e}")

### 2.3 Correlation Matrix
Examine pairwise correlations among all variables. High correlations among predictors may signal multicollinearity, which we will formally test via VIF.

In [ ]:
# ── Figure 2: Correlation heatmap ───────────────────────────────────────────
corr = monthly.corr()
fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5,
            cbar_kws={"shrink": 0.8}, ax=ax)
ax.set_title("Pairwise Correlation Matrix — Monthly Data", weight="bold", fontsize=13)
plt.tight_layout()
plt.show()

# Show correlations with WTI
print("=== Correlations with WTI % Change (sorted) ===")
print(corr["wti_chg_pct"].drop("wti_chg_pct").sort_values(ascending=False).round(4).to_string())

---
## 3. Stationarity Verification — Augmented Dickey-Fuller Test

Following Brooks (Ch. 7), we must verify that all series are stationary before estimating regression or ARMA-type models. Non-stationary series can produce spurious regressions with inflated R² and misleading t-statistics. The ADF test is applied to every variable; the null hypothesis is that the series contains a unit root (is non-stationary).

In [ ]:
# ── ADF test on all monthly series ──────────────────────────────────────────
print("=== Augmented Dickey-Fuller Tests (Monthly) ===")
print(f"{'Variable':<22} {'ADF Stat':>10} {'p-value':>10} {'Lags':>6}  Result")
print("-" * 70)

adf_results = {}
for col in monthly.columns:
    series = monthly[col].dropna()
    if len(series) < 20:
        continue
    result = adfuller(series, autolag="AIC")
    adf_results[col] = {"stat": result[0], "pval": result[1], "lags": result[2]}
    status = "Stationary" if result[1] < 0.05 else "NON-STATIONARY"
    marker = "  ***" if result[1] >= 0.05 else ""
    print(f"{col:<22} {result[0]:>10.3f} {result[1]:>10.4f} {result[2]:>6d}  {status}{marker}")

print("\nNote: Series marked *** may require differencing or should be treated")
print("as levels (rates) rather than returns in the regression.")

---
## 4. OLS Multiple Regression (Monthly)

### 4.1 Baseline OLS
We begin with a standard OLS multiple regression of WTI monthly % changes on all candidate predictors. Per Brooks (Ch. 4–5), this establishes a baseline and lets us identify which factors carry statistically significant explanatory power before moving to more sophisticated time-series models.

Any variables found non-stationary above will be first-differenced before inclusion.

In [ ]:
# ── Prepare regression data: difference any non-stationary series ────────────
reg_monthly = monthly.copy()

# Difference non-stationary level variables (rates / levels that failed ADF)
# tcmsy10_rate, refin_util_rate, crude_stk_excl_spr, us_crude_prod are levels
level_vars = ["tcmsy10_rate", "refin_util_rate", "crude_stk_excl_spr", "us_crude_prod"]
for var in level_vars:
    if var in adf_results and adf_results[var]["pval"] >= 0.05:
        reg_monthly[var + "_diff"] = reg_monthly[var].diff()
        reg_monthly = reg_monthly.drop(columns=[var])
        print(f"Differenced {var} -> {var}_diff (was non-stationary)")
    else:
        print(f"Kept {var} as-is (stationary, p={adf_results.get(var, {}).get('pval', 'N/A')})")

# Drop rows with NaN after differencing
target = "wti_chg_pct"
features = [c for c in reg_monthly.columns if c != target]
reg_data = reg_monthly[[target] + features].dropna()
print(f"\nRegression sample: {len(reg_data)} observations, {len(features)} predictors")

In [ ]:
# ── OLS Regression ──────────────────────────────────────────────────────────
X = sm.add_constant(reg_data[features])
y = reg_data[target]

ols_model = sm.OLS(y, X).fit()
print(ols_model.summary())

### 4.2 Variance Inflation Factors (VIF)
Per Brooks (Ch. 5), multicollinearity inflates standard errors and can make individually insignificant coefficients appear jointly significant. VIF > 10 is a common rule-of-thumb for problematic multicollinearity.

In [ ]:
# ── VIF calculation ─────────────────────────────────────────────────────────
X_no_const = reg_data[features].dropna()
vif_data = pd.DataFrame({
    "Variable": X_no_const.columns,
    "VIF": [variance_inflation_factor(X_no_const.values, i) for i in range(X_no_const.shape[1])]
}).sort_values("VIF", ascending=False)
print("=== Variance Inflation Factors ===")
print(vif_data.to_string(index=False))
high_vif = vif_data[vif_data["VIF"] > 10]
if len(high_vif) > 0:
    print(f"\nWarning: {len(high_vif)} variable(s) with VIF > 10 — consider removal.")
else:
    print("\nNo severe multicollinearity detected (all VIF < 10).")

### 4.3 OLS Residual Diagnostics
Per Brooks (Ch. 5, 8), OLS assumes i.i.d. errors. We test for:
1. **Autocorrelation** — Durbin-Watson statistic and Breusch-Godfrey LM test
2. **Heteroskedasticity** — Breusch-Pagan test and White's test
3. **ARCH effects** — Engle's ARCH-LM test (motivates GARCH if significant)
4. **Normality** — Jarque-Bera on residuals

In [ ]:
# ── OLS Residual Diagnostic Tests ───────────────────────────────────────────
resids = ols_model.resid

print("=" * 60)
print("OLS RESIDUAL DIAGNOSTIC TESTS")
print("=" * 60)

# 1. Durbin-Watson
from statsmodels.stats.stattools import durbin_watson
dw = durbin_watson(resids)
print(f"\n1. Durbin-Watson: {dw:.4f}  (≈2 = no autocorrelation)")

# 2. Breusch-Godfrey serial correlation LM test
from statsmodels.stats.diagnostic import acorr_breusch_godfrey
bg_stat, bg_p, _, _ = acorr_breusch_godfrey(ols_model, nlags=4)
print(f"2. Breusch-Godfrey (4 lags): stat={bg_stat:.3f}, p={bg_p:.4f}")
print(f"   {'Reject H0: serial correlation detected' if bg_p < 0.05 else 'Fail to reject H0: no serial correlation'}")

# 3. Breusch-Pagan heteroskedasticity test
bp_stat, bp_p, _, _ = het_breuschpagan(resids, X)
print(f"3. Breusch-Pagan: stat={bp_stat:.3f}, p={bp_p:.4f}")
print(f"   {'Reject H0: heteroskedasticity detected' if bp_p < 0.05 else 'Fail to reject H0: homoskedastic'}")

# 4. ARCH-LM test (Engle, 1982)
arch_stat, arch_p, _, _ = het_arch(resids, nlags=4)
print(f"4. ARCH-LM (4 lags): stat={arch_stat:.3f}, p={arch_p:.4f}")
print(f"   {'Reject H0: ARCH effects detected → GARCH warranted' if arch_p < 0.05 else 'Fail to reject H0: no ARCH effects'}")

# 5. Jarque-Bera on residuals
jb_stat, jb_p = jarque_bera(resids)
print(f"5. Jarque-Bera (residuals): stat={jb_stat:.2f}, p={jb_p:.4e}")
print(f"   {'Reject H0: residuals non-normal' if jb_p < 0.05 else 'Fail to reject H0: residuals normal'}")

# 6. Ljung-Box on residuals
lb = acorr_ljungbox(resids, lags=[5, 10, 15], return_df=True)
print(f"\n6. Ljung-Box Q-test on residuals:")
print(lb.to_string())

In [ ]:
# ── Figure 3: OLS Residual Plots ────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Residuals over time
axes[0, 0].plot(reg_data.index, resids, linewidth=0.7, color="steelblue")
axes[0, 0].axhline(0, color="red", linestyle="--", linewidth=0.6)
axes[0, 0].set_title("OLS Residuals Over Time", weight="bold")
axes[0, 0].set_ylabel("Residual")
axes[0, 0].grid(True, alpha=0.3)

# Histogram of residuals
axes[0, 1].hist(resids, bins=40, color="steelblue", edgecolor="black", alpha=0.7, density=True)
mu_r, sig_r = resids.mean(), resids.std()
xr = np.linspace(mu_r - 4*sig_r, mu_r + 4*sig_r, 200)
axes[0, 1].plot(xr, (1/(sig_r*np.sqrt(2*np.pi))) * np.exp(-0.5*((xr-mu_r)/sig_r)**2),
                color="red", linewidth=2)
axes[0, 1].set_title("Histogram of Residuals", weight="bold")
axes[0, 1].grid(True, alpha=0.3)

# ACF of residuals
plot_acf(resids, lags=20, ax=axes[1, 0])
axes[1, 0].set_title("ACF of OLS Residuals", weight="bold")

# Squared residuals (visual check for ARCH effects)
axes[1, 1].plot(reg_data.index, resids**2, linewidth=0.7, color="steelblue")
axes[1, 1].set_title("Squared Residuals (Volatility Clustering Check)", weight="bold")
axes[1, 1].set_ylabel("Residual²")
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 4.4 Reduced OLS — Retain Only Significant Predictors
Following the general-to-specific approach (Brooks Ch. 5), we re-estimate the model dropping insignificant variables to improve parsimony and potentially reduce multicollinearity, while retaining economic interpretability.

In [ ]:
# ── Reduced model: keep variables significant at 10% level ──────────────────
sig_vars = [v for v in features if ols_model.pvalues.get(v, 1.0) < 0.10]
print(f"Significant predictors (p < 0.10): {sig_vars}\n")

X_reduced = sm.add_constant(reg_data[sig_vars])
ols_reduced = sm.OLS(y, X_reduced).fit()
print(ols_reduced.summary())

# Compare AIC/BIC
print(f"\nModel comparison:")
print(f"  Full model:    AIC={ols_model.aic:.2f}, BIC={ols_model.bic:.2f}, Adj-R²={ols_model.rsquared_adj:.4f}")
print(f"  Reduced model: AIC={ols_reduced.aic:.2f}, BIC={ols_reduced.bic:.2f}, Adj-R²={ols_reduced.rsquared_adj:.4f}")

---
## 5. Autocorrelation Structure — ACF and PACF Analysis

Per Brooks (Ch. 6), the ACF and PACF of the dependent variable and OLS residuals guide the choice of AR/MA lag order for the mean equation. We examine both the raw WTI returns and the OLS residuals.

In [ ]:
# ── Figure 4: ACF / PACF of WTI returns and OLS residuals ───────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

plot_acf(monthly["wti_chg_pct"].dropna(), lags=20, ax=axes[0, 0])
axes[0, 0].set_title("ACF of WTI Monthly Returns", weight="bold")

plot_pacf(monthly["wti_chg_pct"].dropna(), lags=20, ax=axes[0, 1])
axes[0, 1].set_title("PACF of WTI Monthly Returns", weight="bold")

plot_acf(resids, lags=20, ax=axes[1, 0])
axes[1, 0].set_title("ACF of OLS Residuals", weight="bold")

plot_pacf(resids, lags=20, ax=axes[1, 1])
axes[1, 1].set_title("PACF of OLS Residuals", weight="bold")

plt.tight_layout()
plt.show()

# Information criteria to pick AR order for residuals
print("=== AIC/BIC for AR(p) on OLS residuals ===")
for p in range(0, 5):
    try:
        ar_model = sm.tsa.ARIMA(resids, order=(p, 0, 0)).fit()
        print(f"  AR({p}): AIC={ar_model.aic:.2f}, BIC={ar_model.bic:.2f}")
    except Exception:
        pass

---
## 6. ARX-GARCH(1,1) Model — Monthly

Per Brooks (Ch. 8–9), when ARCH effects are present in the OLS residuals, coefficient standard errors from OLS are unreliable. A GARCH(1,1) variance specification corrects for time-varying volatility, yielding consistent and efficient estimates.

The model specification:
- **Mean equation**: WTI_t = c + φ₁·WTI_{t-1} + β'X_t + ε_t  (ARX with significant regressors)
- **Variance equation**: σ²_t = ω + α₁·ε²_{t-1} + β₁·σ²_{t-1}

We use Student-t innovations to accommodate the fat tails observed in the return distribution.

If the ARCH-LM test above did not reject, we still estimate the GARCH model for comparison — but the OLS results remain the primary inference.

In [ ]:
# ── ARX-GARCH(1,1) with significant exogenous regressors ────────────────────
# Scale returns to percentage points for numerical stability (arch library preference)
y_pct = reg_data[target] * 100
exog = reg_data[sig_vars] * 100  # scale exogenous to match

# Fit AR(1)-GARCH(1,1) with exogenous variables and Student-t distribution
garch_spec = arch_model(
    y_pct,
    x=exog,
    mean="ARX",
    lags=1,
    vol="GARCH",
    p=1, q=1,
    dist="t",
)
garch_fit = garch_spec.fit(disp="off")
print(garch_fit.summary())

### 6.1 GARCH Residual Diagnostics
Per Brooks (Ch. 9), a well-specified GARCH model should produce standardized residuals that are approximately white noise — no remaining autocorrelation in either the standardized residuals or their squares.

In [ ]:
# ── GARCH standardized residual diagnostics ─────────────────────────────────
std_resid = garch_fit.resid / garch_fit.conditional_volatility
std_resid_clean = std_resid.dropna()

# Figure 5: Standardized residual plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Time series
axes[0, 0].plot(std_resid_clean.index, std_resid_clean, linewidth=0.7, color="steelblue")
axes[0, 0].axhline(0, color="red", linestyle="--", linewidth=0.6)
axes[0, 0].set_title("Standardized Residuals Over Time", weight="bold")
axes[0, 0].set_ylabel("Std. Residual")
axes[0, 0].grid(True, alpha=0.3)

# Histogram
axes[0, 1].hist(std_resid_clean, bins=40, color="steelblue", edgecolor="black",
                alpha=0.7, density=True)
axes[0, 1].set_title("Histogram of Standardized Residuals", weight="bold")
axes[0, 1].grid(True, alpha=0.3)

# ACF of standardized residuals
plot_acf(std_resid_clean, lags=20, ax=axes[1, 0])
axes[1, 0].set_title("ACF of Standardized Residuals", weight="bold")

# ACF of squared standardized residuals (check for remaining ARCH effects)
plot_acf(std_resid_clean**2, lags=20, ax=axes[1, 1])
axes[1, 1].set_title("ACF of Squared Std. Residuals", weight="bold")

plt.tight_layout()
plt.show()

# Ljung-Box on standardized residuals and their squares
print("=== Ljung-Box Tests on GARCH Standardized Residuals ===")
lb_resid = acorr_ljungbox(std_resid_clean, lags=[5, 10, 15], return_df=True)
print("\nStandardized Residuals:")
print(lb_resid.to_string())

lb_sq = acorr_ljungbox(std_resid_clean**2, lags=[5, 10, 15], return_df=True)
print("\nSquared Standardized Residuals (ARCH effects check):")
print(lb_sq.to_string())

# ARCH-LM on standardized residuals
arch_lm_post = het_arch(std_resid_clean, nlags=4)
print(f"\nARCH-LM on std. residuals: stat={arch_lm_post[0]:.3f}, p={arch_lm_post[1]:.4f}")
print(f"{'Remaining ARCH effects detected' if arch_lm_post[1] < 0.05 else 'No remaining ARCH effects — model adequate'}")

In [ ]:
# ── Figure 6: Conditional Volatility from GARCH ────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
cond_vol = garch_fit.conditional_volatility / 100  # scale back to decimal
ax.plot(cond_vol.index, cond_vol, color="darkorange", linewidth=1.2, label="GARCH(1,1) Cond. Volatility")
ax.fill_between(cond_vol.index, 0, cond_vol, alpha=0.2, color="darkorange")
ax.set_title("Monthly Conditional Volatility of WTI Returns — GARCH(1,1)", weight="bold")
ax.set_ylabel("Conditional Std. Dev.")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Persistence
alpha = garch_fit.params.get("alpha[1]", 0)
beta = garch_fit.params.get("beta[1]", 0)
print(f"Volatility persistence (α₁ + β₁): {alpha + beta:.4f}")
print(f"Half-life of volatility shocks: {np.log(2) / -np.log(alpha + beta):.1f} periods" if alpha + beta < 1 else "Non-stationary variance")

---
## 7. Weekly Analysis

We now repeat the full analysis pipeline — OLS, diagnostics, and GARCH — on weekly data constructed from daily closing prices. This allows us to compare whether higher-frequency data produces stronger or weaker predictive relationships, and whether the same factors that matter monthly also matter weekly.

Note: Monthly-only variables (GPR, INDPRO, IGREA, US crude production) are excluded from the weekly specification.

In [ ]:
# ── Weekly: Descriptive stats and missing data ──────────────────────────────
print("=== Weekly Descriptive Statistics ===\n")
print(weekly.describe().round(4).to_string())
print(f"\n=== Missing values ===\n{weekly.isnull().sum()}")
print(f"\nTotal obs: {len(weekly)}, Complete cases: {weekly.dropna().shape[0]}")

In [ ]:
# ── Weekly: ADF Tests ───────────────────────────────────────────────────────
print("=== Augmented Dickey-Fuller Tests (Weekly) ===")
print(f"{'Variable':<24} {'ADF Stat':>10} {'p-value':>10} {'Lags':>6}  Result")
print("-" * 72)

w_adf = {}
for col in weekly.columns:
    series = weekly[col].dropna()
    if len(series) < 20:
        continue
    result = adfuller(series, autolag="AIC")
    w_adf[col] = {"stat": result[0], "pval": result[1]}
    status = "Stationary" if result[1] < 0.05 else "NON-STATIONARY ***"
    print(f"{col:<24} {result[0]:>10.3f} {result[1]:>10.4f} {result[2]:>6d}  {status}")

In [ ]:
# ── Weekly: Prepare regression data ─────────────────────────────────────────
reg_weekly = weekly.copy()

# Difference non-stationary level/rate variables
w_level_vars = ["tcmsy10_rate", "refin_util_rate"]
for var in w_level_vars:
    if var in reg_weekly.columns and var in w_adf and w_adf[var]["pval"] >= 0.05:
        reg_weekly[var + "_diff"] = reg_weekly[var].diff()
        reg_weekly = reg_weekly.drop(columns=[var])
        print(f"Differenced {var}")

w_target = "wti_chg_pct"
w_features = [c for c in reg_weekly.columns if c != w_target]

# Drop any remaining rows with NaN
w_reg_data = reg_weekly[[w_target] + w_features].dropna()
print(f"\nWeekly regression sample: {len(w_reg_data)} obs, {len(w_features)} predictors")
print(f"Features: {w_features}")

# ── Weekly OLS ──────────────────────────────────────────────────────────────
w_X = sm.add_constant(w_reg_data[w_features])
w_y = w_reg_data[w_target]
w_ols = sm.OLS(w_y, w_X).fit()
print(w_ols.summary())

In [ ]:
# ── Weekly: OLS Diagnostics ─────────────────────────────────────────────────
w_resids = w_ols.resid

print("=" * 60)
print("WEEKLY OLS RESIDUAL DIAGNOSTIC TESTS")
print("=" * 60)

w_dw = durbin_watson(w_resids)
print(f"\n1. Durbin-Watson: {w_dw:.4f}")

w_bg_stat, w_bg_p, _, _ = acorr_breusch_godfrey(w_ols, nlags=4)
print(f"2. Breusch-Godfrey (4 lags): stat={w_bg_stat:.3f}, p={w_bg_p:.4f}")
print(f"   {'Serial correlation detected' if w_bg_p < 0.05 else 'No serial correlation'}")

w_bp_stat, w_bp_p, _, _ = het_breuschpagan(w_resids, w_X)
print(f"3. Breusch-Pagan: stat={w_bp_stat:.3f}, p={w_bp_p:.4f}")
print(f"   {'Heteroskedasticity detected' if w_bp_p < 0.05 else 'Homoskedastic'}")

w_arch_stat, w_arch_p, _, _ = het_arch(w_resids, nlags=4)
print(f"4. ARCH-LM (4 lags): stat={w_arch_stat:.3f}, p={w_arch_p:.4f}")
print(f"   {'ARCH effects detected → GARCH warranted' if w_arch_p < 0.05 else 'No ARCH effects'}")

w_jb_stat, w_jb_p = jarque_bera(w_resids)
print(f"5. Jarque-Bera: stat={w_jb_stat:.2f}, p={w_jb_p:.4e}")

# Residual plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes[0, 0].plot(w_reg_data.index, w_resids, linewidth=0.5, color="steelblue")
axes[0, 0].axhline(0, color="red", linestyle="--", linewidth=0.6)
axes[0, 0].set_title("Weekly OLS Residuals Over Time", weight="bold")
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].hist(w_resids, bins=60, color="steelblue", edgecolor="black", alpha=0.7)
axes[0, 1].set_title("Histogram of Weekly Residuals", weight="bold")
axes[0, 1].grid(True, alpha=0.3)

plot_acf(w_resids, lags=20, ax=axes[1, 0])
axes[1, 0].set_title("ACF of Weekly OLS Residuals", weight="bold")

axes[1, 1].plot(w_reg_data.index, w_resids**2, linewidth=0.5, color="steelblue")
axes[1, 1].set_title("Squared Weekly Residuals", weight="bold")
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Weekly: Reduced OLS + ARX-GARCH(1,1) ───────────────────────────────────
# Reduced model with significant variables
w_sig_vars = [v for v in w_features if w_ols.pvalues.get(v, 1.0) < 0.10]
print(f"Weekly significant predictors (p < 0.10): {w_sig_vars}\n")

w_X_red = sm.add_constant(w_reg_data[w_sig_vars])
w_ols_red = sm.OLS(w_y, w_X_red).fit()
print(w_ols_red.summary())

print(f"\nWeekly model comparison:")
print(f"  Full:    AIC={w_ols.aic:.2f}, BIC={w_ols.bic:.2f}, Adj-R²={w_ols.rsquared_adj:.4f}")
print(f"  Reduced: AIC={w_ols_red.aic:.2f}, BIC={w_ols_red.bic:.2f}, Adj-R²={w_ols_red.rsquared_adj:.4f}")

In [ ]:
# ── Weekly: ARX-GARCH(1,1) ──────────────────────────────────────────────────
w_y_pct = w_reg_data[w_target] * 100
w_exog_pct = w_reg_data[w_sig_vars] * 100

w_garch_spec = arch_model(
    w_y_pct,
    x=w_exog_pct,
    mean="ARX",
    lags=1,
    vol="GARCH",
    p=1, q=1,
    dist="t",
)
w_garch_fit = w_garch_spec.fit(disp="off")
print(w_garch_fit.summary())

# Standardized residual diagnostics
w_std_resid = (w_garch_fit.resid / w_garch_fit.conditional_volatility).dropna()

print("\n=== Weekly GARCH Ljung-Box Tests ===")
w_lb = acorr_ljungbox(w_std_resid, lags=[5, 10, 15], return_df=True)
print("\nStd. Residuals:")
print(w_lb.to_string())
w_lb_sq = acorr_ljungbox(w_std_resid**2, lags=[5, 10, 15], return_df=True)
print("\nSquared Std. Residuals:")
print(w_lb_sq.to_string())

w_arch_post = het_arch(w_std_resid, nlags=4)
print(f"\nARCH-LM post-GARCH: stat={w_arch_post[0]:.3f}, p={w_arch_post[1]:.4f}")

# Persistence
w_alpha = w_garch_fit.params.get("alpha[1]", 0)
w_beta = w_garch_fit.params.get("beta[1]", 0)
print(f"Weekly volatility persistence (α₁+β₁): {w_alpha + w_beta:.4f}")

---
## 8. Monthly vs. Weekly Comparison

We now directly compare the two frequency specifications to assess whether the predictive relationships are robust across horizons and which granularity better captures WTI price dynamics.

In [ ]:
# ── Summary comparison table ────────────────────────────────────────────────
comparison = pd.DataFrame({
    "Metric": [
        "Observations",
        "OLS Adj. R²",
        "OLS AIC (reduced)",
        "OLS BIC (reduced)",
        "Durbin-Watson",
        "ARCH-LM p-value",
        "GARCH α₁+β₁ (persistence)",
        "GARCH Log-Likelihood",
        "Significant Predictors",
    ],
    "Monthly": [
        len(reg_data),
        f"{ols_reduced.rsquared_adj:.4f}",
        f"{ols_reduced.aic:.2f}",
        f"{ols_reduced.bic:.2f}",
        f"{dw:.4f}",
        f"{arch_p:.4f}",
        f"{alpha + beta:.4f}",
        f"{garch_fit.loglikelihood:.2f}",
        ", ".join(sig_vars),
    ],
    "Weekly": [
        len(w_reg_data),
        f"{w_ols_red.rsquared_adj:.4f}",
        f"{w_ols_red.aic:.2f}",
        f"{w_ols_red.bic:.2f}",
        f"{w_dw:.4f}",
        f"{w_arch_p:.4f}",
        f"{w_alpha + w_beta:.4f}",
        f"{w_garch_fit.loglikelihood:.2f}",
        ", ".join(w_sig_vars),
    ],
})
print("=" * 80)
print("MONTHLY vs. WEEKLY COMPARISON")
print("=" * 80)
print(comparison.to_string(index=False))

In [ ]:
# ── Figure 7: Side-by-side conditional volatility ──────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

m_vol = garch_fit.conditional_volatility / 100
axes[0].plot(m_vol.index, m_vol, color="darkorange", linewidth=1)
axes[0].fill_between(m_vol.index, 0, m_vol, alpha=0.2, color="darkorange")
axes[0].set_title("Monthly GARCH(1,1) Conditional Volatility", weight="bold")
axes[0].set_ylabel("Cond. Std. Dev.")
axes[0].grid(True, alpha=0.3)

w_vol = w_garch_fit.conditional_volatility / 100
axes[1].plot(w_vol.index, w_vol, color="teal", linewidth=0.7)
axes[1].fill_between(w_vol.index, 0, w_vol, alpha=0.2, color="teal")
axes[1].set_title("Weekly GARCH(1,1) Conditional Volatility", weight="bold")
axes[1].set_ylabel("Cond. Std. Dev.")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 9. Discussion & Conclusion

### Summary of Findings

**Hypothesis Revisited:** We tested whether macroeconomic and financial variables can predict monthly and weekly percentage changes in WTI crude oil prices over the period December 1990 – December 2025.

**Methodology Recap:**
1. OLS multiple regression established a baseline, followed by the general-to-specific approach to isolate significant predictors.
2. Residual diagnostics (Durbin-Watson, Breusch-Godfrey, Breusch-Pagan, ARCH-LM, Jarque-Bera) tested the classical OLS assumptions per Brooks Ch. 4–5.
3. Where ARCH effects were present, we re-estimated using an ARX(1)-GARCH(1,1) with Student-t innovations (Brooks Ch. 8–9), which provides heteroskedasticity-consistent inference.
4. Both monthly and weekly frequencies were analyzed to test robustness.

**Key Takeaways** (to be filled in after running the notebook):
- The significant predictors at monthly frequency will reveal which macro/financial channels most strongly transmit to oil prices.
- The GARCH variance equation captures the well-documented volatility clustering in crude oil markets.
- Comparison of monthly vs. weekly will show whether higher-frequency data improves or dilutes the signal — and whether the same factors remain significant, indicating a robust economic relationship rather than a frequency-specific artifact.

### Future Research
- Asymmetric GARCH extensions (EGARCH, GJR-GARCH) to capture leverage effects in oil returns.
- Out-of-sample forecasting evaluation using rolling-window re-estimation.
- Structural break tests (Chow, CUSUM) to assess parameter stability across regimes (e.g., 2008 crisis, COVID-19, post-2020 energy transition).